# Satellite Remote Sensing of Urban Heat Islands Using MODIS Aqua MYD11A2: A Case Study on Lagos State, Nigeria

## Overview

This project analyses seasonal fluctuations in Land Surface Temperature (LST) across Lagos State, Nigeria, from 2013 to 2022 using MODIS Aqua MYD11A2 data, and examines the correlation between LST and NO2 tropospheric column density (OMI AURA). Landsat 8 imagery was used to assess land use/land cover (LULC) change over the same period, and gridded population data was used to map the population most at risk from Urban Heat Island (UHI) effects.

**Study Area:** Lagos State, Nigeria (6°25'N–6°33'N, 2°25'E–4°22'E)  
**Duration:** Jan 2023 – Aug 2023  
**Role:** Solo project (MSc Dissertation, University of Aberdeen)  
**Status:** Completed

---

## Methods & Tools

**Data Sources**

- MODIS Aqua MYD11A2 V6.1 Land Surface Temperature/Emissivity 8-Day L3 Global 1km — NASA Earthdata
- OMI AURA NO2 Tropospheric Column (OMNO2d) — NASA Earthdata
- Landsat 8 OLI/TIRS (Path 191, Rows 055/056) — USGS Earth Explorer
- GRID3 Nigeria Population Estimates — WorldPop, University of Southampton

**Processing Steps**

1. Retrieve and reproject MODIS LST granules (2013–2022); rescale digital numbers to °C using the split-window algorithm output
2. Apply Focal Statistics to fill LST data gaps caused by cloud cover
3. Aggregate LST and NO2 tropospheric column data on a 30×30 fishnet grid and compute Pearson's correlation coefficient
4. Classify Landsat 8 imagery (2013 vs 2022) into four LULC classes using a Support Vector Machine classifier and validate with a confusion matrix
5. Run Zonal Statistics against gridded population data to map the population at risk of UHI by Local Government Area (LGA)

**Tools Used**

| Tool | Purpose |
|------|---------|
| ArcGIS Pro / ArcGIS 10.8 | Raster processing, Focal Statistics, Zonal Statistics, LULC classification |
| Python (Spyder IDE, arcpy) | Batch retrieval and processing of MODIS/OMI HDF granules |
| SPSS | Pearson's correlation and scatter plot analysis |
| pandas / matplotlib | Trend analysis and visualisation |
| folium | Interactive mapping of population risk by LGA |


## Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import folium

## Data

LST summary statistics by season (2013–2022), extracted from MODIS Aqua MYD11A2 processing (Tables 4.1 and 4.2).

In [ ]:
years = list(range(2013, 2023))

rainy_season = pd.DataFrame({
    "year": years,
    "lst_max": [31.75, 33.92, 32.48, 33.57, 33.86, 32.21, 33.11, 33.07, 33.24, 30.89],
    "lst_min": [22.13, 20.44, 20.23, 21.89, 9.29, 16.03, 21.90, 22.10, 20.93, 18.22],
    "lst_mean": [25.41, 25.50, 25.22, 26.55, 25.04, 25.26, 25.91, 26.48, 26.47, 24.76],
})

dry_season = pd.DataFrame({
    "year": years,
    "lst_max": [31.30, 31.24, 30.96, 30.74, 31.15, 31.66, 32.49, 31.53, 31.96, 31.79],
    "lst_min": [23.24, 23.38, 23.47, 22.65, 23.72, 23.91, 23.97, 23.90, 24.16, 23.79],
    "lst_mean": [26.03, 26.37, 26.13, 26.47, 26.74, 26.84, 27.16, 26.88, 27.18, 27.12],
})

rainy_season.head()

In [ ]:
no2 = pd.DataFrame({
    "year": years,
    "no2_dry_mean": [1.55, 1.41, 1.71, 1.80, 1.70, 1.76, 1.64, 1.64, 1.65, 1.62],
    "no2_rainy_mean": [1.10, 1.07, 1.07, 1.12, 1.02, 1.09, 1.09, 0.94, 1.04, 1.04],
})
no2.head()

## Analysis

### Summary statistics

In [ ]:
rainy_season[["lst_max", "lst_min", "lst_mean"]].describe()

In [ ]:
dry_season[["lst_max", "lst_min", "lst_mean"]].describe()

### LST trend chart (Rainy vs Dry Season, 2013–2022)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(rainy_season["year"], rainy_season["lst_mean"], marker="o", label="Rainy Season Mean LST", color="steelblue")
ax.plot(dry_season["year"], dry_season["lst_mean"], marker="s", label="Dry Season Mean LST", color="coral")
ax.set_title("Mean Land Surface Temperature by Season (2013–2022)", fontsize=14, fontweight="bold")
ax.set_xlabel("Year")
ax.set_ylabel("LST (°C)")
ax.legend()
ax.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

### NO2 tropospheric column trend chart

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(no2["year"], no2["no2_dry_mean"], marker="o", label="Dry Season", color="steelblue")
ax.plot(no2["year"], no2["no2_rainy_mean"], marker="s", label="Rainy Season", color="coral")
ax.set_title("NO2 Tropospheric Column Concentrations (2013–2022)", fontsize=14, fontweight="bold")
ax.set_xlabel("Year")
ax.set_ylabel("NO2 Concentration (10^15 molecules/cm^2)")
ax.legend()
ax.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

## Spatial Distribution

Interactive map of the eight Lagos LGAs identified as critical risk areas for UHI impact, based on the Zonal Statistics population-at-risk analysis (approximate LGA centroid coordinates shown for illustration).

In [ ]:
critical_risk_lgas = pd.DataFrame({
    "name":  ["Ifako-Ijaiye", "Agege", "Ikeja", "Oshodi-Isolo", "Mushin", "Shomolu", "Surulere", "Ajeromi-Ifelodun"],
    "lat":   [6.6739, 6.6018, 6.6018, 6.5556, 6.5286, 6.5401, 6.5060, 6.4557],
    "lon":   [3.2903, 3.3221, 3.3515, 3.3079, 3.3548, 3.3839, 3.3547, 3.3325],
    "risk":  ["Critical"] * 8,
})

m = folium.Map(
    location=[critical_risk_lgas["lat"].mean(), critical_risk_lgas["lon"].mean()],
    zoom_start=11,
    tiles="CartoDB positron",
)

for _, row in critical_risk_lgas.iterrows():
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=10,
        color="firebrick",
        fill=True,
        fill_opacity=0.7,
        tooltip=f"{row['name']}: {row['risk']} Risk",
        popup=folium.Popup(f"<b>{row['name']}</b><br>UHI Risk: {row['risk']}", max_width=200),
    ).add_to(m)

m

## Key Findings

- LST during the rainy season showed a general downward trend (2013–2022): maximum LST fell from 31.75°C to 30.89°C, while mean LST declined from 25.41°C to 24.76°C
- LST during the dry season showed the opposite pattern, with an overall warming trend of approximately 1.09°C in mean temperature from 2013 to 2022
- LST and NO2 tropospheric column density were positively and significantly correlated in both seasons (p < 0.001), with the relationship consistently stronger during the dry season (R² up to 0.539) than the rainy season (R² as low as 0.099)
- Built-up areas increased by 12.63% and vegetation by 5.93% between 2013 and 2022, while wetlands declined by 24.26% and water bodies by 1.23%
- Eight LGAs — Surulere, Agege, Ifako-Ijaiye, Ikeja, Shomolu, Oshodi-Isolo, Ajeromi-Ifelodun, and Mushin — were identified as critical-risk areas for population exposure to UHI effects

---

## Links

[View Data Source](https://urs.earthdata.nasa.gov/)
